# 03 — Baseline Qwen

Exécuter depuis la racine du dépôt. Les résultats ne sont valides que si les cellules sont réellement exécutées.

In [ ]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/'data').exists():
    ROOT=Path('/content/EduLab-AI-Version2')
print(ROOT)

In [ ]:
!pip -q install -r requirements-colab.txt

In [ ]:
MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, json, time
tok=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype='auto', device_map='auto')

In [ ]:
# Construire 30 prompts équilibrés (10 par classe) depuis le test
rows=[json.loads(x) for x in open(ROOT/'data/processed/edulab_teacher_test.jsonl',encoding='utf-8')]
prompts=[]
for cls in ['Troisième','Terminale C','Terminale D']:
 prompts += [r for r in rows if r['class_name']==cls][:10]
assert len(prompts)==30
out=[]
for r in prompts:
 text=tok.apply_chat_template([{'role':'user','content':r['instruction']+'\nContexte: '+r['context']}],tokenize=False,add_generation_prompt=True)
 inputs=tok(text,return_tensors='pt').to(model.device); t=time.time()
 ids=model.generate(**inputs,max_new_tokens=160,do_sample=False)
 out.append({**{k:r[k] for k in ['id','class_name','subject','task']},'prediction':tok.decode(ids[0][inputs.input_ids.shape[1]:],skip_special_tokens=True),'latency_s':time.time()-t})
with open(ROOT/'reports/baseline_predictions.jsonl','w',encoding='utf-8') as f:
 for x in out:f.write(json.dumps(x,ensure_ascii=False)+'\n')
print(len(out))